# VARUNA - Run in Colab

This notebook runs the **VARUNA** uncertainty-aware flood early-warning system end-to-end in Google Colab.

**Pipeline:** ERA5/GFS/IMD/GloFAS/Sentinel-NRT -> RainCast rainfall -> HydroNet discharge -> FloodMap flood probability -> RiskEngine decisions -> Gemini plain-language brief.

### Setup
1. Click **Runtime → Change runtime type → GPU** (T4 / L4 recommended).
2. Set your Gemini API key as a Colab secret:
   - Go to **Tools → Secrets**
   - Add `GEMINI_API_KEY` with your key from https://aistudio.google.com/app/apikey
   - Or just enter it at the prompt below.

> No Copernicus/Sentinel credentials? The pipeline degrades gracefully with placeholders.

In [ ]:
# Step 1 — Install dependencies (torch CPU build for Colab free tier)
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
%pip install --quiet numpy pyyaml requests einops scipy tqdm matplotlib \
                    netCDF4 xarray h5netcdf pandas
%pip install --quiet google-generativeai
print('✅ Dependencies installed')

In [ ]:
# Step 2 — Clone the VARUNA repo and install the package
import os, sys

# Remove existing clone (if any) and re-clone for a clean state
if os.path.exists('VARUNA'):
    !rm -rf VARUNA
!git clone --depth 1 https://github.com/Amarnath10i/VARUNA.git
%cd VARUNA

# Install as editable package
%pip install --quiet -e . --no-deps  # dependencies already installed above

print('✅ VARUNA cloned and installed')

In [ ]:
# Step 3 — Set the Gemini API key
# Recommended: use Colab Secrets (Tools → Secrets → add GEMINI_API_KEY)
# Fallback: enter the key at the prompt

try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('✅ Gemini API key loaded from Colab secrets')
except Exception:
    api_key = input('Enter your Gemini API key: ')
    os.environ['GEMINI_API_KEY'] = api_key.strip()
    print('✅ Gemini API key set')

In [ ]:
# Step 4 — Run smoke tests
# Validates metrics, uncertainty, model forward passes, and the pipeline — no heavy training or network needed

!python -m pytest tests/test_smoke.py -v 2>&1 | tail -20

# Or run directly:
# !python tests/test_smoke.py

In [ ]:
# Step 5 — Run the end-to-end forecast pipeline
# Generates risk decisions for all configured regions and a plain-language brief

from varuna.pipeline.forecast import ForecastPipeline
from pathlib import Path

cfg = Path('config/config.yaml')
pipe = ForecastPipeline(cfg, run_dir='runs/colab_forecast')
decisions = pipe.run()

import json
print(json.dumps(decisions, indent=2))

In [ ]:
# Step 6 — Display the Gemini-generated risk brief
# Re-generates from the saved decisions using the Gemini API

from varuna.ai.gemini import GeminiRiskBrief
from pathlib import Path
import json

out = json.loads(Path('runs/colab_forecast/latest_cycle.json').read_text())
brief = GeminiRiskBrief().generate(out['decisions'])
print(brief)

### What just happened?

1. **Dependencies** — Installed PyTorch, NumPy, xarray, and `google-generativeai`.
2. **Repo** — Cloned VARUNA from GitHub and installed it as a local package.
3. **API key** — Loaded your Gemini key securely (via Colab secrets or runtime input).
4. **Tests** — Ran smoke tests covering metrics, conformal UQ, model forward passes, and the pipeline.
5. **Forecast** — Executed the full `ForecastPipeline` (live feeds → RainCast → HydroNet → FloodMap → RiskEngine).
6. **Brief** — Generated a plain-language risk briefing with Gemini.

The output JSON and brief are saved to `runs/colab_forecast/` in Colab's VM (download via file browser).

---
To save your work: `File → Save a copy in GitHub` and commit to a new branch.